# Pipeline Anotasi Buzzer — notebook penjalan

Notebook ini **hanya memanggil** tahap-tahap di paket `annotation/` (logika, tes, dan invarian ada di sana).
Hasil tersimpan di `state/` dan `out/`, sama seperti lewat CLI, sehingga notebook dan CLI bisa dipakai bergantian.

**Cara pakai**
- Jalankan sel dari atas ke bawah. Berhenti di setiap **GATE** (pekerjaan manual di Label Studio), lalu lanjutkan.
- Selalu jalankan **sel Persiapan** lebih dulu setiap kali kernel dimulai.
- Menjalankan ulang sel aman: tahap yang sudah selesai menolak ditimpa atau melanjutkan dari titik terakhir.
- Bila sebuah tahap berhenti, pesan `BERHENTI: ...` menjelaskan penyebab dan langkah perbaikannya.
- **Jangan menulis API key di sel** (output tersimpan di berkas notebook). Sel API key memakai input tersembunyi.
- Jangan mengedit `pedoman_anotasi.md`, `rules.yaml`, prompt yang sudah dievaluasi, atau berkas di `state/`.

Panduan lengkap: `README.md`.

## 0. Persiapan

In [ ]:
%load_ext autoreload
%autoreload 2   # muat ulang annotation/*.py yang berubah tanpa restart kernel
import os, sys, json
from pathlib import Path

USE_MOCK = False   # True hanya untuk mencoba alur dengan model tiruan (menulis ke state_mock/, bukan state/)
BASE = None        # folder berisi config.yaml; None = folder paket annotation/

# folder yang memuat paket annotation/ (dicari dari folder kerja kernel ke atas; cadangan: lokasi proyek)
CODE_DIR = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "annotation" / "__init__.py").exists()),
                Path(r"C:\Users\ASUS\Documents\Scriptshiz\code\preprocess"))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if USE_MOCK:
    os.environ["ANNOTATION_MOCK"] = "1"
else:
    os.environ.pop("ANNOTATION_MOCK", None)        # mencegah mode tiruan tertinggal di kernel

import pandas as pd
from IPython.display import Markdown, display
from annotation import (a0_validate_input, a1_sample_subset, a1b_import_and_split, a2_eval_prompt, a2b_fewshot_candidates,
                        a3_annotate_full, a4_export_review, a4b_import_review, a5_finalize, demo, llm, probe, status)
from annotation.a2_eval_prompt import read_rounds
from annotation.common import AnnotationError, load_config, paths, read_csv_str, sha256_file

pd.set_option("display.max_colwidth", 80)
P = paths(load_config(BASE))
EXPORTS = P.state.parent / "exports"          # simpan ekspor Label Studio di sini
EXPORTS.mkdir(exist_ok=True)


def jalankan(fn, *args):
    """Jalankan satu tahap dengan config terbaru; tampilkan pesan BERHENTI alih-alih traceback."""
    try:
        return fn(load_config(BASE), *args)
    except AnnotationError as e:
        print("BERHENTI:", e)


def biaya(harga_input=0.25, harga_cached=0.025, harga_output=2.00):
    """Ringkasan penggunaan per tahap dari respons API + perkiraan biaya (USD per 1 juta token;
    harga GPT-5 Mini dicek 2026-09-20). completion_tokens sudah termasuk reasoning_tokens."""
    u = llm.usage_summary(llm.read_calls(load_config(BASE)))
    if not len(u):
        return print("Belum ada panggilan API.")
    cached = u.cached_tokens.fillna(0)
    u["usd"] = ((u.prompt_tokens - cached) * harga_input + cached * harga_cached + u.completion_tokens * harga_output) / 1e6
    display(u.round({"cached_pct": 1, "reasoning_rata2": 1, "usd": 4}))


print("Mode:", "TIRUAN (state_mock/)" if USE_MOCK else "PRODUKSI (state/)", "| folder kerja:", CODE_DIR)
jalankan(status.run)

## D. Demo pelabelan (opsional, tanpa membekukan apa pun)

Untuk mencoba/mendemokan pelabelan di Label Studio sebelum pedoman final. Akun demo dipilih dari luar subset resmi
(dijamin tidak akan masuk 63 akun GATE 1), memakai template yang sama dengan GATE 1, dan **tidak** menulis ke `state/`,
tidak mengisi manifest, dan tidak butuh `guideline.sha256`. Label demo tidak dipakai di alur resmi.

1. Jalankan sel di bawah -> berkas di `annotation/demo/`.
2. Label Studio: proyek baru **DEMO - pelabelan**; *Labeling Setup -> Custom template -> Code*: tempel isi
   `annotation/demo/label_studio_config_demo.xml`; *Data Import*: `annotation/demo/demo_tasks.json`; *Save*.
3. Labeli (KU, IP1-IP4, label), *Export -> JSON*, simpan sebagai `annotation/exports/demo.json`.
4. Jalankan sel pemeriksaan: menampilkan apakah tiap label sesuai Tabel 4 (pada alur resmi, label yang tidak sesuai
   akan ditolak), tanpa memblokir.

In [ ]:
DEMO_N = 12            # jumlah akun demo
_ = jalankan(demo.build, DEMO_N)

In [ ]:
DEMO_EXPORT = EXPORTS / "demo.json"
hasil_demo = jalankan(demo.check, DEMO_EXPORT)
if hasil_demo is not None:
    display(hasil_demo)

## 1. Bekukan pedoman (sekali, sebelum GATE 1)

1. Pastikan `annotation/pedoman_anotasi.md` sudah versi final (salinan persis pedoman Anda).
2. Jalankan sel di bawah, lalu salin hash yang dicetak ke `config.yaml` → `guideline: sha256:`.
3. Setelah itu pedoman **tidak boleh diubah** (invarian I8).

In [ ]:
cfg = load_config(BASE)
print("Hash pedoman saat ini :", sha256_file(P.guideline))
print("guideline.sha256 config:", cfg["guideline"]["sha256"])
print("Cocok" if cfg["guideline"]["sha256"] == sha256_file(P.guideline) else "BELUM cocok -- salin hash di atas ke config.yaml")

## 2. A0 + A1 — validasi data dan ambil subset ±10%

In [ ]:
_ = jalankan(a0_validate_input.run)

In [ ]:
_ = jalankan(a1_sample_subset.run)     # sekaligus membekukan rules.yaml (Tabel 4)

## ⏸ GATE 1 — pelabelan manual subset (Label Studio)

1. Jalankan Label Studio di terminal PowerShell: `.\annotation\start_label_studio.ps1`, lalu buka http://localhost:8080.
2. Buat proyek **GATE 1**. *Labeling Setup → Code*: tempel isi `annotation/state/label_studio_config_gate1.xml`.
3. *Import*: `annotation/state/manual_tasks.json` (tanpa pra-anotasi LLM).
4. Untuk setiap akun isi **KU**, **IP1–IP4** (boleh kosong bila KU = `tidak`), dan **label** yang sesuai Tabel 4.
   Jangan *Skip* dan jangan submit dua kali.
5. *Export → JSON* (bukan JSON-MIN), simpan sebagai `annotation/exports/gate1.json`.

## 3. A1b — impor label manual dan bagi tuning/validation

In [ ]:
GATE1_EXPORT = EXPORTS / "gate1.json"
_ = jalankan(a1b_import_and_split.run, GATE1_EXPORT)

In [ ]:
if P.manual_subset.exists():
    m = read_csv_str(P.manual_subset)
    display(pd.crosstab(m.split, m.manual_label, margins=True, margins_name="total"))
else:
    print("manual_subset.csv belum ada -- selesaikan GATE 1 dan A1b dulu.")

## ⏸ GATE 2 — tulis prompt v1

Isi `annotation/prompts/prompt_v1.yaml` (hapus semua `TODO`): `system`, `instruction`, `criteria` (ku, ip1–ip4),
dan `decision_rule: "{{rules}}"` (teks aturan dibuat otomatis dari `rules.yaml`). `response_format` dan `options`
biarkan apa adanya; `fewshot_ref: null`. Jangan memasukkan teks akun validation set.

Sel berikut menampilkan **pesan persis yang akan dikirim** untuk satu akun, tanpa memanggil API.
Tata letak `static_system` (default): bagian beku di pesan system (identik antar-akun, bisa di-*cache*),
data akun saja di pesan user.

In [ ]:
VERSION = "v1"
cfg = load_config(BASE)
clean = pd.read_csv(P.clean_data, dtype={"user_id": str, "caption_text": str}, keep_default_na=False)
try:
    prompt = llm.load_prompt(cfg, VERSION, clean)
    msgs = llm.render_messages(clean.iloc[0], prompt, cfg)
    for msg in msgs:
        print(f"===== {msg['role'].upper()} =====\n{msg['content']}\n")
    frozen = msgs[0]["content"] if prompt.layout == "static_system" else ""
    print(f"prompt_sha256 = {prompt.sha256[:16]}... | layout = {prompt.layout} | format = {prompt.render_format}")
    print(f"perkiraan token per panggilan: {sum(llm.count_tokens(x['content']) for x in msgs)}"
          + (f" | bagian beku (system): {llm.count_tokens(frozen)} -- prefiks >= 1.024 token baru bisa di-cache "
             f"(perkiraan tokenizer, bukan hitungan API; ukur lewat cached_tokens)" if frozen else ""))
    if prompt.rule_warning:
        print("PERINGATAN:", prompt.rule_warning)
except AnnotationError as e:
    print("BERHENTI:", e)

### API key dan uji parameter (sekali per sesi kernel)

Key dimasukkan lewat input tersembunyi dan hanya hidup selama kernel berjalan.

In [ ]:
from getpass import getpass


def _key_valid(k):
    return k.startswith("sk-") and not any(ch.isspace() for ch in k)


key = os.getenv("OPENAI_API_KEY", "")
if not _key_valid(key):                       # belum ada, atau yang tersimpan bukan key OpenAI (mis. teks lain tertempel)
    key = getpass("Tempel OPENAI_API_KEY (diawali sk-), lalu Enter: ").strip()
if _key_valid(key):
    os.environ["OPENAI_API_KEY"] = key
    print(f"API key terpasang untuk sesi ini (sk-...{key[-4:]}).")
else:
    os.environ.pop("OPENAI_API_KEY", None)
    print("Yang ditempel bukan API key OpenAI (harus diawali 'sk-', tanpa spasi/baris baru). "
          "Salin ulang key dari platform.openai.com, lalu jalankan sel ini lagi.")

In [ ]:
# Opsional (sekali saja): uji parameter yang diterima GPT-5 Mini, ±10 panggilan pendek tanpa data akun.
# Isi llm.reasoning_effort di config.yaml hanya dengan nilai yang DITERIMA (atau biarkan null).
# jalankan(probe.run)

## 4. Loop optimasi — A2 (evaluasi) dan A2b (kandidat few-shot)

**A2** melabeli 63 akun subset dengan prompt `VERSION` dan menghitung recall buzzer.
- Recall validasi ≥ 0,85 → prompt **dibekukan**, lanjut ke bagian 5.
- Belum → jalankan **A2b**, buat versi berikutnya, ganti `VERSION`, jalankan A2 lagi.
- Setelah v4 (v1 + 3 revisi) prompt dibekukan otomatis.
- Bila ada akun gagal (error API/parse), jalankan ulang sel A2: akun yang sudah selesai tidak dipanggil lagi.

In [ ]:
VERSION = "v1"        # ganti ke v2, v3, v4 pada putaran berikutnya
_ = jalankan(a2_eval_prompt.run, VERSION)

In [ ]:
r = read_rounds(load_config(BASE))
if len(r):
    tab = r[r.repeat == 1].pivot_table(index=["round", "prompt_version"], columns="set",
                                       values=["positives", "tp", "recall"], aggfunc="first")
    display(tab.round(3))
biaya()

In [ ]:
out = jalankan(a2b_fewshot_candidates.run)
if out:
    display(pd.read_csv(out, dtype=str))

### ⏸ GATE 2 (putaran berikutnya) — buat versi baru

1. Salin `prompt_v1.yaml` → `prompt_v2.yaml` (jangan mengubah versi yang sudah dievaluasi); perjelas kriteria yang
   sering keliru menurut kolom `diff_fields` di atas.
2. Opsional: buat `prompts/fewshot_v2.json` berisi akun **tuning** dari tabel kandidat, mis.
   `[{"user_id": "..."}, {"user_id": "..."}]`, lalu isi `fewshot_ref: fewshot_v2.json` di `prompt_v2.yaml`.
3. Pratinjau dengan sel GATE 2 di atas (ubah `VERSION`), lalu jalankan sel A2 dengan `VERSION = "v2"`.

## 5. A3 — label seluruh akun dengan prompt beku (±626 panggilan)

In [ ]:
_ = jalankan(a3_annotate_full.run)     # jalankan ulang bila ada akun gagal, sampai 626/626
biaya()

## 6. A4 — ekspor akun buzzer-LLM (di luar subset) untuk ditinjau

In [ ]:
_ = jalankan(a4_export_review.run)

## ⏸ GATE 3 — tinjauan manual (Label Studio)

1. Buat proyek **baru** GATE 3 (terpisah dari GATE 1). *Code*: tempel `annotation/state/label_studio_config_gate3.xml`.
2. *Import*: `annotation/state/review_tasks.json`. Label LLM tampil sebagai pra-anotasi; KU/IP LLM disembunyikan.
   Bila pra-anotasi tidak muncul, aktifkan opsi pra-label dari prediksi di pengaturan proyek.
3. Untuk setiap akun isi KU dan IP sendiri, lalu konfirmasi atau koreksi label sesuai Tabel 4.
4. *Export → JSON*, simpan sebagai `annotation/exports/gate3.json`.

## 7. A4b — impor tinjauan

In [ ]:
GATE3_EXPORT = EXPORTS / "gate3.json"
_ = jalankan(a4b_import_review.run, GATE3_EXPORT)

## 8. A5 — label akhir dan laporan

In [ ]:
hasil = jalankan(a5_finalize.run)
if hasil is not None:
    display(hasil.label_source.value_counts().to_frame("akun"))
    display(Markdown(P.report_md.read_text(encoding="utf-8")))

In [ ]:
jalankan(status.run)
biaya()